# 01. Dataset de modelagem — snapshots de contratos de obra

Este notebook constrói `dataset_modelagem_obras_snapshots.csv`.

## Regra de conclusão do projeto

A **data definitiva de conclusão** passa a ser a **data de recebimento provisório** informada em `vw_data_receb_prov.xlsx`:

`data_conclusao_definitiva = data_recebimento_provisorio`

A coluna `Data_conclusao` de `vw_bi_contratos` é preservada apenas para auditoria e **não é usada para definir o target**.

Regras principais:

- o universo elegível é definido **exclusivamente por `vw_bi_contratos.csv`**;
- a data de recebimento provisório é incorporada por `Id_contrato = idContrato`;
- datas sentinela como `1900-01-01`, valores equivalentes e `"NULL"` são tratados como ausentes;
- cada contrato gera snapshots mensais desde a `Data_OS` até `data_conclusao_definitiva` ou `DATA_REFERENCIA`;
- parcelas, liberações e aditivos são fontes auxiliares de features;
- toda feature usa somente eventos com data `<= data_corte`;
- aditivos futuros não alteram retrospectivamente as features de snapshots antigos;
- o prazo final usado no target considera aditivos assinados até `data_conclusao_definitiva`;
- `target_dias_atraso = max(0, data_conclusao_definitiva - data_termino_vigente_final)`;
- `target_atraso = 1` quando `target_dias_atraso > 0`;
- contratos sem `data_conclusao_definitiva` permanecem disponíveis para scoring, mas sem target;
- `flag_apto_treino = 1` somente para snapshots anteriores à conclusão definitiva e com target conhecido.

> Para compatibilidade com os notebooks de modelagem já existentes, `data_conclusao_controle` será mantida como alias de `data_conclusao_definitiva`.


In [4]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

FILES = {
    "contratos": "vw_bi_contratos.csv",
    "parcelas": "vw_bi_parcelas.csv",
    "liberacoes": "vw_tparcela_liberacao.csv",
    "aditivos_valor": "vw_taditivovalor.csv",
    "aditivos_prazo": "vw_taditivoprazo.csv",
    "recebimentos_provisorios": "vw_data_receb_prov.csv",
}

# Data usada para encerrar snapshots de contratos ainda sem
# data_conclusao_definitiva.
DATA_REFERENCIA = pd.Timestamp("2026-09-29")


## 1. Pastas e carregamento

In [5]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate
    return current

ROOT = find_project_root()

if (ROOT / "data" / "raw").exists():
    RAW = ROOT / "data" / "raw"
    PROCESSED = ROOT / "data" / "processed"
    QUALITY = ROOT / "reports" / "data_quality"
else:
    # Fallback útil quando o notebook e os CSVs estão na mesma pasta.
    RAW = ROOT
    PROCESSED = ROOT / "data" / "processed"
    QUALITY = ROOT / "reports" / "data_quality"

for folder in [PROCESSED, QUALITY]:
    folder.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW:", RAW)
print("PROCESSED:", PROCESSED)


ROOT: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras
RAW: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\raw
PROCESSED: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(RAW / name, low_memory=False, encoding="utf-8-sig")


def read_excel(name: str) -> pd.DataFrame:
    return pd.read_excel(RAW / name)


contratos = read_csv(FILES["contratos"])
parcelas = read_csv(FILES["parcelas"])
liberacoes = read_csv(FILES["liberacoes"])
aditivos_valor = read_csv(FILES["aditivos_valor"])
aditivos_prazo = read_csv(FILES["aditivos_prazo"])
recebimentos_provisorios = read_csv(FILES["recebimentos_provisorios"])

datasets = {
    "contratos": contratos,
    "parcelas": parcelas,
    "liberacoes": liberacoes,
    "aditivos_valor": aditivos_valor,
    "aditivos_prazo": aditivos_prazo,
    "recebimentos_provisorios": recebimentos_provisorios,
}

manifest = pd.DataFrame([
    {
        "dataset": key,
        "arquivo": filename,
        "linhas": len(datasets[key]),
        "colunas": datasets[key].shape[1],
        "tamanho_bytes": (RAW / filename).stat().st_size,
        "sha256": sha256(RAW / filename),
    }
    for key, filename in FILES.items()
])

manifest.to_csv(
    QUALITY / "manifesto_carga_modelagem.csv",
    index=False,
    encoding="utf-8-sig",
)
display(manifest)


ValueError: Excel file format cannot be determined, you must specify an engine manually.

## 2. Padronização de tipos e datas sentinela

In [ ]:
def to_dates(df, columns):
    result = df.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_datetime(result[column], errors="coerce")
            result.loc[result[column].dt.year <= 1900, column] = pd.NaT
    return result


def to_numbers(df, columns):
    result = df.copy()
    for column in columns:
        if column in result.columns:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    return result


def parse_data_recebimento_provisorio(series: pd.Series) -> pd.Series:
    """Normaliza datas vindas do Excel sem confundir Timestamp com serial numérico."""

    def parse_one(value):
        if pd.isna(value):
            return pd.NaT

        # pd.read_excel normalmente já devolve as células formatadas como data.
        if isinstance(value, (pd.Timestamp, np.datetime64)):
            return pd.Timestamp(value)

        # Se a planilha vier sem formatação de data, aceita o serial do Excel.
        if isinstance(value, (int, float, np.integer, np.floating)):
            return (
                pd.Timestamp("1899-12-30")
                + pd.to_timedelta(float(value), unit="D")
            )

        text = str(value).strip()
        if text.lower() in {"", "null", "none", "nat", "nan"}:
            return pd.NaT

        return pd.to_datetime(text, errors="coerce")

    parsed = pd.to_datetime(series.map(parse_one), errors="coerce")
    parsed.loc[parsed.dt.year <= 1900] = pd.NaT
    return parsed


contratos = to_numbers(
    contratos,
    ["Id_contrato", "Valor_total_contrato", "Prazo_total", "Id_fornecedor"],
)
contratos = to_dates(
    contratos,
    [
        "Data_assinatura_contrato", "Data_conclusao_contrato", "Data_conclusao",
        "Data_proposta", "Data_Orcamento", "Data_OS",
        "Data_Paralisacao", "Data_Prev_Retomada",
    ],
)

parcelas = to_numbers(
    parcelas,
    ["id_parcela", "id_contrato", "valor_reajustado", "ordem_status", "valor_ja_liberado"],
)
parcelas = to_dates(
    parcelas,
    ["data_prevista_parcela", "data_efetiva", "data_liberacao"],
)

liberacoes = to_numbers(
    liberacoes,
    ["id_Contrato", "id_Parcela", "Abono_atraso", "Atraso_considerado", "Valor_total_liberado"],
)
liberacoes = to_dates(
    liberacoes,
    ["Data_efet_conclusao", "Data_liberação"],
)

aditivos_valor = to_numbers(aditivos_valor, ["idContrato"])
aditivos_valor = to_dates(aditivos_valor, ["Data_assinatura_TA_valor"])

aditivos_prazo = to_numbers(aditivos_prazo, ["idContrato", "Aditado_em_dias"])
aditivos_prazo = to_dates(
    aditivos_prazo,
    ["Data_termino_anterior", "Data_termino_nova", "Data_assinatura_ta_prazo"],
)

# Nova fonte de conclusão definitiva
recebimentos_provisorios = to_numbers(
    recebimentos_provisorios,
    ["idContrato"],
)
recebimentos_provisorios["data_conclusao_definitiva"] = (
    parse_data_recebimento_provisorio(
        recebimentos_provisorios["data_recebimento_provisorio"]
    )
)

if recebimentos_provisorios["idContrato"].duplicated().any():
    duplicados = recebimentos_provisorios.loc[
        recebimentos_provisorios["idContrato"].duplicated(keep=False),
        ["idContrato", "data_recebimento_provisorio", "data_conclusao_definitiva"],
    ]
    raise ValueError(
        "Há idContrato duplicado em vw_data_receb_prov.xlsx. "
        "A regra de consolidação precisa ser definida antes de prosseguir.\n"
        f"{duplicados.head(20)}"
    )

# Mantém apenas chave + data tratada e incorpora ao cadastro mestre.
recebimentos_merge = recebimentos_provisorios[
    ["idContrato", "data_conclusao_definitiva"]
].copy()

contratos = contratos.merge(
    recebimentos_merge,
    how="left",
    left_on="Id_contrato",
    right_on="idContrato",
    validate="one_to_one",
)

contratos.drop(columns=["idContrato"], inplace=True)


## 3. Universo de contratos de obra

`vw_bi_contratos` define o universo elegível. Registros existentes apenas nas demais fontes são descartados antes da construção das features.


In [ ]:
summary = pd.DataFrame([
    {
        "dataset": name,
        "linhas": len(df_),
        "colunas": df_.shape[1],
        "duplicidades_exatas": int(df_.duplicated().sum()),
        "percentual_nulos": round(float(df_.isna().mean().mean() * 100), 2),
    }
    for name, df_ in {
        "contratos": contratos,
        "parcelas": parcelas,
        "liberacoes": liberacoes,
        "aditivos_valor": aditivos_valor,
        "aditivos_prazo": aditivos_prazo,
        "recebimentos_provisorios": recebimentos_provisorios,
    }.items()
])
display(summary)

ids_obras = set(
    pd.to_numeric(
        contratos["Id_contrato"],
        errors="coerce",
    ).dropna().astype(int)
)

# Fontes auxiliares ficam restritas ao universo de contratos de obra.
parcelas = parcelas[parcelas["id_contrato"].isin(ids_obras)].copy()
liberacoes = liberacoes[liberacoes["id_Contrato"].isin(ids_obras)].copy()
aditivos_valor = aditivos_valor[aditivos_valor["idContrato"].isin(ids_obras)].copy()
aditivos_prazo = aditivos_prazo[aditivos_prazo["idContrato"].isin(ids_obras)].copy()

cobertura_recebimento = pd.DataFrame({
    "metrica": [
        "contratos_de_obra",
        "contratos_com_data_os_valida",
        "contratos_com_data_conclusao_definitiva",
        "contratos_com_data_os_e_conclusao_definitiva",
        "concluidos_no_status_sem_data_definitiva",
    ],
    "valor": [
        len(ids_obras),
        int(contratos["Data_OS"].notna().sum()),
        int(contratos["data_conclusao_definitiva"].notna().sum()),
        int(
            (
                contratos["Data_OS"].notna()
                & contratos["data_conclusao_definitiva"].notna()
            ).sum()
        ),
        int(
            (
                contratos["Status_Contrato"].eq("CONCLUÍDO")
                & contratos["data_conclusao_definitiva"].isna()
            ).sum()
        ),
    ],
})

display(cobertura_recebimento)

# Validação de coerência temporal.
datas_invalidas = contratos[
    contratos["data_conclusao_definitiva"].notna()
    & contratos["Data_OS"].notna()
    & (contratos["data_conclusao_definitiva"] < contratos["Data_OS"])
]

if len(datas_invalidas):
    raise ValueError(
        "Foram encontradas datas de recebimento provisório anteriores à Data_OS."
    )

print(f"Contratos de obra no cadastro: {len(ids_obras)}")
print(f"Contratos com Data_OS válida: {contratos['Data_OS'].notna().sum()}")
print(
    "Contratos com data_conclusao_definitiva:",
    contratos["data_conclusao_definitiva"].notna().sum(),
)


,dataset,linhas,colunas,duplicidades_exatas,percentual_nulos
0,contratos,87,26,0,16.3600
1,parcelas,13010,11,8,1.5300
2,liberacoes,11918,8,1,0.0000
3,aditivos_valor,271,10,6,1.0000
4,aditivos_prazo,245,10,0,0.0400


Contratos de obra no cadastro: 87
Contratos com Data_OS válida: 78


## 4. Calendário mensal de snapshots

Para cada obra com `Data_OS` válida:

1. o primeiro snapshot é a própria `Data_OS`;
2. são incluídos os últimos dias de cada mês;
3. se houver `data_conclusao_definitiva`, o último snapshot é essa data;
4. se ainda não houver recebimento provisório, o calendário segue até `DATA_REFERENCIA`.

Assim, `Data_conclusao` do cadastro antigo não encerra mais o calendário.


In [ ]:
calendar_rows = []

for _, contrato in contratos[contratos["Data_OS"].notna()].iterrows():
    inicio = contrato["Data_OS"]

    fim_natural = (
        contrato["data_conclusao_definitiva"]
        if pd.notna(contrato["data_conclusao_definitiva"])
        else DATA_REFERENCIA
    )
    fim = min(fim_natural, DATA_REFERENCIA)

    if fim < inicio:
        continue

    cortes = list(pd.date_range(inicio, fim, freq="ME"))

    if not cortes or cortes[0] > inicio:
        cortes = [inicio] + cortes

    if cortes[-1] != fim:
        cortes.append(fim)

    for data_corte in sorted(set(cortes)):
        calendar_rows.append(
            (int(contrato["Id_contrato"]), data_corte)
        )

calendar = pd.DataFrame(
    calendar_rows,
    columns=["id_contrato", "data_corte"],
)

assert not calendar.duplicated(
    ["id_contrato", "data_corte"]
).any()

print(f"Snapshots no calendário: {len(calendar):,}")
print(f"Contratos no calendário: {calendar['id_contrato'].nunique():,}")
display(calendar.head())


Snapshots no calendário: 1,435
Contratos no calendário: 78


,id_contrato,data_corte
0,11,2019-07-24
1,11,2019-07-31
2,11,2019-08-31
3,11,2019-09-30
4,11,2019-10-31


## 5. Features e targets

Para cada `id_contrato + data_corte`, as features usam somente eventos conhecidos até a fotografia.

### Data definitiva de conclusão

`data_conclusao_definitiva` é a **data de recebimento provisório** de `vw_data_receb_prov.xlsx`.

### Prazo final de referência

Para contratos com conclusão definitiva conhecida, o prazo contratual final considera apenas aditivos de prazo cuja assinatura ocorreu até `data_conclusao_definitiva`.

### Targets

- `target_dias_atraso = max(0, data_conclusao_definitiva - data_termino_vigente_final)`
- `target_atraso = 1` quando `target_dias_atraso > 0`, senão `0`

Contratos sem recebimento provisório permanecem sem target.

Os campos terminados em `_controle`, além de `data_conclusao_definitiva`, são de auditoria e **não devem entrar em X**.


In [ ]:
contratos_idx = contratos.set_index("Id_contrato")

rows = []

for cid, data_corte in calendar.itertuples(index=False, name=None):
    contrato = contratos_idx.loc[cid]

    # Prazo original
    data_termino_original = (
        contrato["Data_OS"] + pd.to_timedelta(float(contrato["Prazo_total"]), unit="D")
        if pd.notna(contrato["Prazo_total"])
        else pd.NaT
    )

    # Aditivos de prazo conhecidos até o snapshot
    api = aditivos_prazo[
        (aditivos_prazo["idContrato"] == cid)
        & aditivos_prazo["Data_assinatura_ta_prazo"].notna()
        & (aditivos_prazo["Data_assinatura_ta_prazo"] <= data_corte)
    ].sort_values("Data_assinatura_ta_prazo")

    if api["Data_termino_nova"].notna().any():
        data_termino_vigente_corte = api["Data_termino_nova"].dropna().iloc[-1]
    else:
        data_termino_vigente_corte = data_termino_original

    # Parcelas do contrato
    pi = parcelas[parcelas["id_contrato"] == cid]

    parcelas_previstas = pi[
        pi["data_prevista_parcela"].notna()
        & (pi["data_prevista_parcela"] <= data_corte)
    ]

    parcelas_concluidas = pi[
        pi["data_efetiva"].notna()
        & (pi["data_efetiva"] <= data_corte)
    ]

    atrasos = (
        parcelas_concluidas["data_efetiva"]
        - parcelas_concluidas["data_prevista_parcela"]
    ).dt.days.dropna()

    parcelas_90d = parcelas_concluidas[
        parcelas_concluidas["data_efetiva"]
        >= data_corte - pd.Timedelta(days=90)
    ]
    atrasos_90d = (
        parcelas_90d["data_efetiva"]
        - parcelas_90d["data_prevista_parcela"]
    ).dt.days.dropna()

    # Liberações conhecidas até o snapshot
    li = liberacoes[
        (liberacoes["id_Contrato"] == cid)
        & liberacoes["Data_liberação"].notna()
        & (liberacoes["Data_liberação"] <= data_corte)
    ]

    li_90d = li[
        li["Data_liberação"] >= data_corte - pd.Timedelta(days=90)
    ]

    # Aditivos de valor conhecidos até o snapshot
    avi = aditivos_valor[
        (aditivos_valor["idContrato"] == cid)
        & aditivos_valor["Data_assinatura_TA_valor"].notna()
        & (aditivos_valor["Data_assinatura_TA_valor"] <= data_corte)
    ]

    dias_desde_os = (data_corte - contrato["Data_OS"]).days

    prazo_vigente_total_dias = (
        (data_termino_vigente_corte - contrato["Data_OS"]).days
        if pd.notna(data_termino_vigente_corte)
        else np.nan
    )

    pct_prazo_consumido = (
        dias_desde_os / prazo_vigente_total_dias
        if pd.notna(prazo_vigente_total_dias) and prazo_vigente_total_dias > 0
        else np.nan
    )

    valor_total_contrato = pd.to_numeric(
        pd.Series([contrato["Valor_total_contrato"]]), errors="coerce"
    ).iloc[0]

    valor_liberado_acumulado = (
        li["Valor_total_liberado"].sum(min_count=1) if len(li) else 0
    )

    pct_financeiro_liberado = (
        valor_liberado_acumulado / valor_total_contrato
        if pd.notna(valor_total_contrato) and valor_total_contrato != 0
        else np.nan
    )

    # Prazo final usado no target: somente informações formalizadas até a conclusão definitiva
    data_conclusao_definitiva = contrato["data_conclusao_definitiva"]

    if pd.notna(data_conclusao_definitiva):
        aditivos_ate_conclusao = aditivos_prazo[
            (aditivos_prazo["idContrato"] == cid)
            & aditivos_prazo["Data_assinatura_ta_prazo"].notna()
            & (aditivos_prazo["Data_assinatura_ta_prazo"] <= data_conclusao_definitiva)
        ].sort_values("Data_assinatura_ta_prazo")

        if aditivos_ate_conclusao["Data_termino_nova"].notna().any():
            data_termino_vigente_final = (
                aditivos_ate_conclusao["Data_termino_nova"].dropna().iloc[-1]
            )
        else:
            data_termino_vigente_final = data_termino_original
    else:
        data_termino_vigente_final = pd.NaT

    target_dias_atraso = (
        max(0, (data_conclusao_definitiva - data_termino_vigente_final).days)
        if pd.notna(data_conclusao_definitiva) and pd.notna(data_termino_vigente_final)
        else np.nan
    )

    rows.append({
        "id_contrato": cid,
        "data_corte": data_corte,
        "numero_contrato": contrato["Numero_contrato"],
        "classificacao_contrato": contrato["Classificacao_contrato"],
        "id_fornecedor": contrato["Id_fornecedor"],
        "valor_total_contrato": valor_total_contrato,
        "prazo_original_dias": contrato["Prazo_total"],
        "data_os": contrato["Data_OS"],
        "data_termino_original": data_termino_original,
        "data_termino_vigente_corte": data_termino_vigente_corte,
        "dias_desde_os": dias_desde_os,
        "dias_restantes_prazo_vigente": (
            (data_termino_vigente_corte - data_corte).days
            if pd.notna(data_termino_vigente_corte)
            else np.nan
        ),
        "prazo_vigente_total_dias": prazo_vigente_total_dias,
        "pct_prazo_consumido": pct_prazo_consumido,
        "qtd_aditivos_prazo": len(api),
        "dias_aditados_acumulados": (
            api["Aditado_em_dias"].sum(min_count=1) if len(api) else 0
        ),
        "dias_desde_ultimo_aditivo_prazo": (
            (data_corte - api["Data_assinatura_ta_prazo"].max()).days
            if len(api) else np.nan
        ),
        "qtd_aditivos_valor": len(avi),
        "dias_desde_ultimo_aditivo_valor": (
            (data_corte - avi["Data_assinatura_TA_valor"].max()).days
            if len(avi) else np.nan
        ),
        "qtd_parcelas_previstas_ate_corte": len(parcelas_previstas),
        "qtd_parcelas_concluidas_ate_corte": len(parcelas_concluidas),
        "pct_parcelas_concluidas_vs_previstas": (
            len(parcelas_concluidas) / len(parcelas_previstas)
            if len(parcelas_previstas)
            else np.nan
        ),
        "atraso_medio_parcelas_dias": atrasos.mean() if len(atrasos) else np.nan,
        "atraso_mediano_parcelas_dias": atrasos.median() if len(atrasos) else np.nan,
        "atraso_maximo_parcelas_dias": atrasos.max() if len(atrasos) else np.nan,
        "qtd_parcelas_atrasadas": int((atrasos > 0).sum()) if len(atrasos) else 0,
        "pct_parcelas_atrasadas": float((atrasos > 0).mean()) if len(atrasos) else np.nan,
        "atraso_medio_parcelas_90d": atrasos_90d.mean() if len(atrasos_90d) else np.nan,
        "qtd_parcelas_atrasadas_90d": int((atrasos_90d > 0).sum()) if len(atrasos_90d) else 0,
        "qtd_liberacoes": len(li),
        "valor_liberado_acumulado": valor_liberado_acumulado,
        "valor_liberado_90d": (
            li_90d["Valor_total_liberado"].sum(min_count=1) if len(li_90d) else 0
        ),
        "atraso_medio_liberacao_dias": (
            li["Atraso_considerado"].mean() if len(li) else np.nan
        ),
        "atraso_maximo_liberacao_dias": (
            li["Atraso_considerado"].max() if len(li) else np.nan
        ),
        "pct_financeiro_liberado": pct_financeiro_liberado,
        "gap_prazo_financeiro": (
            pct_prazo_consumido - pct_financeiro_liberado
            if pd.notna(pct_prazo_consumido) and pd.notna(pct_financeiro_liberado)
            else np.nan
        ),
        "status_final_controle": contrato["Status_Contrato"],
        "data_conclusao_sistema_controle": contrato["Data_conclusao"],
        "data_conclusao_definitiva": data_conclusao_definitiva,
        # Alias preservado por compatibilidade com os notebooks 03/04.
        "data_conclusao_controle": data_conclusao_definitiva,
        "data_termino_vigente_final_controle": data_termino_vigente_final,
        "target_dias_atraso": target_dias_atraso,
        "target_atraso": (
            int(target_dias_atraso > 0) if pd.notna(target_dias_atraso) else np.nan
        ),
    })

dataset = pd.DataFrame(rows)
dataset.head()


,id_contrato,data_corte,numero_contrato,classificacao_contrato,id_fornecedor,valor_total_contrato,prazo_original_dias,data_os,data_termino_original,data_termino_vigente_corte,dias_desde_os,dias_restantes_prazo_vigente,prazo_vigente_total_dias,pct_prazo_consumido,qtd_aditivos_prazo,dias_aditados_acumulados,dias_desde_ultimo_aditivo_prazo,qtd_aditivos_valor,dias_desde_ultimo_aditivo_valor,qtd_parcelas_previstas_ate_corte,qtd_parcelas_concluidas_ate_corte,pct_parcelas_concluidas_vs_previstas,atraso_medio_parcelas_dias,atraso_mediano_parcelas_dias,atraso_maximo_parcelas_dias,qtd_parcelas_atrasadas,pct_parcelas_atrasadas,atraso_medio_parcelas_90d,qtd_parcelas_atrasadas_90d,qtd_liberacoes,valor_liberado_acumulado,valor_liberado_90d,atraso_medio_liberacao_dias,atraso_maximo_liberacao_dias,pct_financeiro_liberado,gap_prazo_financeiro,status_final_controle,data_conclusao_controle,data_termino_vigente_final_controle,target_dias_atraso,target_atraso
0,11,2019-07-24,160/2018,Reforma Parcial,35,"276,770.8100",150,2019-07-24,2019-12-21,2019-12-21,0,150,150,0.0000,0,0,NaN,1,23.0000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,0.0000,0.0000,NaN,NaN,0.0000,0.0000,CONCLUÍDO,2020-01-15,2019-12-21,25.0000,1.0000
1,11,2019-07-31,160/2018,Reforma Parcial,35,"276,770.8100",150,2019-07-24,2019-12-21,2019-12-21,7,143,150,0.0467,0,0,NaN,1,30.0000,0,0,NaN,NaN,NaN,NaN,0,NaN,NaN,0,0,0.0000,0.0000,NaN,NaN,0.0000,0.0467,CONCLUÍDO,2020-01-15,2019-12-21,25.0000,1.0000
2,11,2019-08-31,160/2018,Reforma Parcial,35,"276,770.8100",150,2019-07-24,2019-12-21,2019-12-21,38,112,150,0.2533,0,0,NaN,1,61.0000,1,1,1.0000,-3.0000,-3.0000,-3.0000,0,0.0000,-3.0000,0,1,"19,716.1800","19,716.1800",0.0000,0.0000,0.0712,0.1821,CONCLUÍDO,2020-01-15,2019-12-21,25.0000,1.0000
3,11,2019-09-30,160/2018,Reforma Parcial,35,"276,770.8100",150,2019-07-24,2019-12-21,2019-12-21,68,82,150,0.4533,0,0,NaN,1,91.0000,2,3,1.5000,-14.3333,-12.0000,-3.0000,0,0.0000,-14.3333,0,3,"59,228.5600","59,228.5600",0.0000,0.0000,0.2140,0.2393,CONCLUÍDO,2020-01-15,2019-12-21,25.0000,1.0000
4,11,2019-10-31,160/2018,Reforma Parcial,35,"276,770.8100",150,2019-07-24,2019-12-21,2019-12-21,99,51,150,0.6600,0,0,NaN,1,122.0000,3,3,1.0000,-14.3333,-12.0000,-3.0000,0,0.0000,-14.3333,0,3,"59,228.5600","59,228.5600",0.0000,0.0000,0.2140,0.4460,CONCLUÍDO,2020-01-15,2019-12-21,25.0000,1.0000


## 6. Faixa de antecedência, elegibilidade de treino e exportação

In [ ]:
dataset["faixa_antecedencia"] = pd.cut(
    dataset["dias_restantes_prazo_vigente"],
    [-np.inf, 29, 89, 179, np.inf],
    labels=["<30d", "30-89d", "90-179d", ">=180d"],
)

dataset["flag_apto_treino"] = (
    dataset["data_conclusao_definitiva"].notna()
    & (dataset["data_corte"] < dataset["data_conclusao_definitiva"])
    & dataset["target_dias_atraso"].notna()
).astype("int8")

dataset = dataset.sort_values(
    ["id_contrato", "data_corte"]
).reset_index(drop=True)

OUTPUT = PROCESSED / "dataset_modelagem_obras_snapshots.csv"
dataset.to_csv(
    OUTPUT,
    index=False,
    encoding="utf-8-sig",
)

print("Arquivo:", OUTPUT)
print(f"Linhas: {len(dataset):,}")
print(f"Colunas: {dataset.shape[1]}")
print(f"Contratos: {dataset['id_contrato'].nunique():,}")
print(
    f"Snapshots aptos para treino: "
    f"{int(dataset['flag_apto_treino'].sum()):,}"
)
print(
    "Contratos com snapshots aptos para treino:",
    dataset.loc[
        dataset["flag_apto_treino"].eq(1),
        "id_contrato",
    ].nunique(),
)


Arquivo: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed\dataset_modelagem_obras_snapshots.csv
Linhas: 1,435
Colunas: 43
Contratos: 78
Snapshots aptos para treino: 1,034
Contratos com snapshots aptos para treino: 59


## 7. Validações de consistência

As validações abaixo confirmam principalmente:

- unicidade de `id_contrato + data_corte`;
- universo restrito aos contratos de obra;
- nenhum snapshot apto ao treino ocorre na própria conclusão definitiva ou depois dela;
- contratos sem recebimento provisório não recebem target supervisionado;
- `data_conclusao_controle` permanece igual a `data_conclusao_definitiva` por compatibilidade.

> As quantidades de snapshots e contratos rotulados podem mudar em relação à versão anterior porque o calendário e os targets agora são definidos pela data de recebimento provisório.


In [ ]:
assert not dataset.duplicated(
    ["id_contrato", "data_corte"]
).any()

assert set(dataset["id_contrato"]).issubset(ids_obras)

# Alias compatível com notebooks posteriores.
assert (
    dataset["data_conclusao_controle"].fillna(pd.Timestamp("1900-01-01"))
    == dataset["data_conclusao_definitiva"].fillna(pd.Timestamp("1900-01-01"))
).all()

# Nenhuma linha apta a treino pode estar na conclusão definitiva ou depois.
assert (
    dataset.loc[
        dataset["flag_apto_treino"].eq(1),
        "data_corte",
    ]
    < dataset.loc[
        dataset["flag_apto_treino"].eq(1),
        "data_conclusao_definitiva",
    ]
).all()

# Contratos sem data definitiva não recebem target.
sem_conclusao_definitiva = dataset[
    "data_conclusao_definitiva"
].isna()

assert dataset.loc[
    sem_conclusao_definitiva,
    "target_dias_atraso",
].isna().all()

assert dataset.loc[
    sem_conclusao_definitiva,
    "target_atraso",
].isna().all()

quality = pd.DataFrame({
    "metrica": [
        "snapshots",
        "colunas",
        "contratos",
        "data_corte_minima",
        "data_corte_maxima",
        "duplicidades_id_data",
        "contratos_com_conclusao_definitiva",
        "snapshots_aptos_treino",
        "contratos_aptos_treino",
        "taxa_atraso_snapshots_treino",
    ],
    "valor": [
        len(dataset),
        dataset.shape[1],
        dataset["id_contrato"].nunique(),
        dataset["data_corte"].min(),
        dataset["data_corte"].max(),
        dataset.duplicated(
            ["id_contrato", "data_corte"]
        ).sum(),
        dataset.loc[
            dataset["data_conclusao_definitiva"].notna(),
            "id_contrato",
        ].nunique(),
        int(dataset["flag_apto_treino"].sum()),
        dataset.loc[
            dataset["flag_apto_treino"].eq(1),
            "id_contrato",
        ].nunique(),
        dataset.loc[
            dataset["flag_apto_treino"].eq(1),
            "target_atraso",
        ].mean(),
    ],
})

quality.to_csv(
    QUALITY / "qualidade_dataset_modelagem_obras_snapshots.csv",
    index=False,
    encoding="utf-8-sig",
)

display(quality)


,metrica,valor
0,snapshots,1435
1,colunas,43
2,contratos,78
3,data_corte_minima,2018-03-22 00:00:00
4,data_corte_maxima,2026-09-29 00:00:00
5,duplicidades_id_data,0
6,snapshots_aptos_treino,1034
7,contratos_aptos_treino,59
8,taxa_atraso_snapshots_treino,0.8395


## 8. Colunas que não devem entrar diretamente no modelo

Excluir de `X`:

- `id_contrato` e `numero_contrato`: identificadores;
- `status_final_controle`;
- `data_conclusao_sistema_controle`: data antiga, mantida apenas para auditoria;
- `data_conclusao_definitiva`: recebimento provisório e componente direto do target;
- `data_conclusao_controle`: alias da conclusão definitiva para compatibilidade;
- `data_termino_vigente_final_controle`;
- `target_dias_atraso`;
- `target_atraso`;
- `flag_apto_treino`;
- datas absolutas cruas, salvo quando deliberadamente transformadas em features.

`data_corte` deve ser preservada para validação temporal e auditoria, mas não necessariamente usada como preditor bruto.

### Observação importante

A antiga `Data_conclusao` de `vw_bi_contratos` não define mais o desfecho do projeto. Ela é exportada como `data_conclusao_sistema_controle` somente para auditoria.


In [ ]:
print("Diretório atual:", Path.cwd().resolve())
print("PROCESSED:", PROCESSED.resolve())
# print("OUTPUT:", OUTPUT.resolve())
# print("Existe:", OUTPUT.exists())

Diretório atual: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\notebooks
PROCESSED: C:\Users\Gabriel\OneDrive\Documentos\projeto_previsao_termino_obras\data\processed
